In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import hopsworks
from datetime import datetime, timedelta

# ==========================================
# 1. PARAMETERS & CONFIGURATION
# ==========================================
CITY_NAME = "Faisalabad"
LATITUDE = 31.4187
LONGITUDE = 73.0791
PROJECT_NAME = "AQI_Predictor_fsd"
HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")

# ==========================================
# 2. DATA FETCHING (Open-Meteo)
# ==========================================
def fetch_weather_and_air_quality(start_date: str, end_date: str) -> pd.DataFrame:
    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
        "timezone": "UTC"
    }
    
    aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    aq_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["pm2_5", "pm10", "nitrogen_dioxide"],
        "timezone": "UTC"
    }

    res_weather = requests.get(weather_url, params=weather_params).json()
    res_aq = requests.get(aq_url, params=aq_params).json()

    df_weather = pd.DataFrame(res_weather["hourly"])
    df_weather["time"] = pd.to_datetime(df_weather["time"])

    df_aq = pd.DataFrame(res_aq["hourly"])
    df_aq["time"] = pd.to_datetime(df_aq["time"])

    df = pd.merge(df_weather, df_aq, on="time", how="inner")
    df["city"] = CITY_NAME
    
    return df

# ==========================================
# 3. FEATURE ENGINEERING FUNCTION
# ==========================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("time").reset_index(drop=True)

    # Time-based features
    df["hour"] = df["time"].dt.hour
    df["day"] = df["time"].dt.day
    df["dayofweek"] = df["time"].dt.dayofweek
    df["month"] = df["time"].dt.month
    df["is_weekend"] = df["dayofweek"].apply(lambda x: 1 if x >= 5 else 0)

    # Cyclical Features
    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24.0)

    # Lag & Rolling Features
    df["pm2_5_lag_1h"] = df["pm2_5"].shift(1)
    df["pm2_5_lag_24h"] = df["pm2_5"].shift(24)
    df["pm2_5_roll_mean_6h"] = df["pm2_5"].shift(1).rolling(window=6).mean()
    df["pm2_5_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(window=24).mean()

    # Derived Features: Change Rates
    df["aqi_change_rate_1h"] = (df["pm2_5_lag_1h"] - df["pm2_5"].shift(2)) / (df["pm2_5"].shift(2) + 1e-5)
    df["aqi_change_rate_24h"] = (df["pm2_5_lag_1h"] - df["pm2_5_lag_24h"]) / (df["pm2_5_lag_24h"] + 1e-5)

    df = df.dropna().reset_index(drop=True)
    df["city"] = df["city"].astype(str)
    # Convert datetime column to timezone-aware UTC ISO format or epoch ms
    df["time"] = pd.to_datetime(df['time']).astype("datetime64[us]").dt.tz_localize("UTC")

    return df

# ==========================================
# 4. HOPSWORKS INGESTION & BACKFILL
# ==========================================
def run_pipeline(backfill: bool = False):
    if not HOPSWORKS_API_KEY:
        raise ValueError("HOPSWORKS_API_KEY environment variable is not set!")

    if backfill:
        print("--- Running Historical Data Backfill ---")
        start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
        end_date = datetime.now().strftime("%Y-%m-%d")
    else:
        print("--- Running Daily Feature Ingestion ---")
        start_date = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")
        end_date = datetime.now().strftime("%Y-%m-%d")

    # Step 1: Fetch Raw Data
    raw_df = fetch_weather_and_air_quality(start_date, end_date)

    # Step 2: Compute Features
    features_df = engineer_features(raw_df)

    # Step 3: Windows-Compatible Hopsworks Connection
    cert_dir = os.path.join(os.getcwd(), "certs")
    os.makedirs(cert_dir, exist_ok=True)

    project = hopsworks.login(
        host="eu-west.cloud.hopsworks.ai",
        port=443,
        project=PROJECT_NAME,
        api_key_value=HOPSWORKS_API_KEY,
        cert_folder=cert_dir
    )
    fs = project.get_feature_store()

    # Step 4: Create or Get Feature Group
    aqi_fg = fs.get_or_create_feature_group(
        name="weather_aqi_hourly",
        version=1,
        primary_key=["city"],
        event_time="time",
        description="Hourly weather metrics, air quality indicators, and temporal/lag features.",
        online_enabled=True
    )
    # Step 5: Insert Features into Hopsworks
    print(f"Upserting {len(features_df)} rows into Hopsworks...")
    aqi_fg.insert(features_df, storage="online")
    print("Ingestion complete!")


In [2]:
if __name__ == "__main__":
    run_pipeline(backfill=True)

--- Running Historical Data Backfill ---


2026-07-26 12:21:16,116 INFO: Initializing external client
2026-07-26 12:21:16,117 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-07-26 12:21:19,648 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42119
Upserting 8760 rows into Hopsworks...


Uploading Dataframe: 100.00% |██████████| Rows 1/1 | Elapsed Time: 00:01 | Remaining Time: 00:00

Ingestion complete!
